# Attention in Transformers: Concepts and Code in PyTorch

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

---
<a id='1'></a>
# Part 1: Self-Attention

Self-attention lets every token look at every other token and decide how much to attend to it.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q (Query):** what this token is looking for
- **K (Key):** what each token offers
- **V (Value):** the actual content each token contributes
- **√d_k:** scaling to prevent softmax saturation

In [11]:
class SelfAttention(nn.Module):

    def __init__(self, d_model=2, row_dim=0, col_dim=1):
        ## d_model = number of embedding values per token
        ## default d_model=2 so we can verify math by hand
        ## in "Attention Is All You Need", d_model=512
        super().__init__()

        ## W_q, W_k, W_v: learned projection matrices, no bias per original paper
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        self.row_dim = row_dim
        self.col_dim = col_dim

    def forward(self, token_encodings):
        ## Step 1: Project inputs into Q, K, V
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)

        ## Step 2: Compute similarity scores: Q @ K^T
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))

        ## Step 3: Scale by sqrt(d_k)
        scaled_sims = sims / torch.tensor(k.size(self.col_dim) ** 0.5)

        ## Step 4: Softmax -> attention weights
        attention_percents = F.softmax(scaled_sims, dim=self.col_dim)

        ## Step 5: Weighted sum of values
        attention_scores = torch.matmul(attention_percents, v)

        return attention_scores

In [12]:
## 3 tokens, each with a 2-dim embedding
encodings_matrix = torch.tensor([[1.16, 0.23],
                                  [0.57, 1.36],
                                  [4.41, -2.16]])

torch.manual_seed(42)

selfAttention = SelfAttention(d_model=2, row_dim=0, col_dim=1)

selfAttention(encodings_matrix)

tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)

In [13]:
print("W_q:\n", selfAttention.W_q.weight.transpose(0, 1))
print("\nW_k:\n", selfAttention.W_k.weight.transpose(0, 1))
print("\nW_v:\n", selfAttention.W_v.weight.transpose(0, 1))

W_q:
 tensor([[ 0.5406, -0.1657],
        [ 0.5869,  0.6496]], grad_fn=<TransposeBackward0>)

W_k:
 tensor([[-0.1549, -0.3443],
        [ 0.1427,  0.4153]], grad_fn=<TransposeBackward0>)

W_v:
 tensor([[ 0.6233,  0.6146],
        [-0.5188,  0.1323]], grad_fn=<TransposeBackward0>)


In [14]:
q = selfAttention.W_q(encodings_matrix)
k = selfAttention.W_k(encodings_matrix)
v = selfAttention.W_v(encodings_matrix)

print("Q:\n", q)
print("\nK:\n", k)
print("\nV:\n", v)

Q:
 tensor([[ 0.7621, -0.0428],
        [ 1.1063,  0.7890],
        [ 1.1164, -2.1336]], grad_fn=<MmBackward0>)

K:
 tensor([[-0.1469, -0.3038],
        [ 0.1057,  0.3685],
        [-0.9914, -2.4152]], grad_fn=<MmBackward0>)

V:
 tensor([[ 0.6038,  0.7434],
        [-0.3502,  0.5303],
        [ 3.8695,  2.4246]], grad_fn=<MmBackward0>)


In [15]:
sims = torch.matmul(q, k.transpose(dim0=0, dim1=1))
print("Similarity scores (Q @ K^T):\n", sims)

Similarity scores (Q @ K^T):
 tensor([[-0.0990,  0.0648, -0.6523],
        [-0.4022,  0.4078, -3.0024],
        [ 0.4842, -0.6683,  4.0461]], grad_fn=<MmBackward0>)


In [16]:
scaled_sims = sims / (torch.tensor(2) ** 0.5)
print("Scaled similarities:\n", scaled_sims)

Scaled similarities:
 tensor([[-0.0700,  0.0458, -0.4612],
        [-0.2844,  0.2883, -2.1230],
        [ 0.3424, -0.4725,  2.8610]], grad_fn=<DivBackward0>)


In [17]:
attention_percents = F.softmax(scaled_sims, dim=1)
print("Attention weights:\n", attention_percents)
print("\nRow sums (should all be 1.0):", attention_percents.sum(dim=1))

Attention weights:
 tensor([[0.3573, 0.4011, 0.2416],
        [0.3410, 0.6047, 0.0542],
        [0.0722, 0.0320, 0.8959]], grad_fn=<SoftmaxBackward0>)

Row sums (should all be 1.0): tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [18]:
output = torch.matmul(attention_percents, selfAttention.W_v(encodings_matrix))
print("Manual output:\n", output)
print("\nMatches forward pass?", torch.allclose(output, selfAttention(encodings_matrix), atol=1e-6))

Manual output:
 tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)

Matches forward pass? True
